# Polars avanzado con gráficos
Notebook autónomo de análisis con Polars y visualización con Matplotlib. Usaremos datos sintéticos reproducibles, sin depender de pandas ni de descargas externas.

## 1. Preparación de un dataset sintético

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

# Dataset determinista generado con Python: no requiere pandas, NumPy ni descargas externas.
n = 150
df = pl.DataFrame({
    'sepal length (cm)': [5.0 + (i % 50) * 0.02 + (i // 50) * 0.7 for i in range(n)],
    'sepal width (cm)': [3.5 - (i % 10) * 0.03 + (i // 50) * 0.1 for i in range(n)],
    'petal length (cm)': [1.4 + (i % 50) * 0.04 + (i // 50) * 1.8 for i in range(n)],
    'petal width (cm)': [0.2 + (i % 25) * 0.01 + (i // 50) * 0.5 for i in range(n)],
    'target': [i // 50 for i in range(n)],
})
df.head()


### Explicación
Generamos un dataset determinista con Python y lo construimos directamente como `pl.DataFrame`, sin descargas ni conversiones desde pandas.


## 2. Estadísticas básicas y exploración

In [ ]:
df.describe()

### Explicación
`df.describe()` en Polars genera estadísticas resumen (count, mean, std, min, max) para columnas numéricas, útil para exploración temprana.


In [ ]:
df['target'].value_counts()

### Explicación
Conteo de valores de la columna `target` para verificar distribución de clases; Polars permite acceso directo con `col['target'].value_counts()`.


### Gráfico: Dispersión Sepal Length vs Sepal Width

In [ ]:
plt.scatter(df['sepal length (cm)'].to_list(), df['sepal width (cm)'].to_list())
plt.xlabel('Sepal Length')
plt.ylabel('Sepal Width')
plt.title('Scatter Plot - Sepal Dimensions')


### Explicación
Matplotlib recibe listas nativas mediante `to_list()`. Convertir a pandas solo sería necesario si una API externa lo exigiera; no es necesario para graficar desde Polars.


## 3. GroupBy avanzado con agregaciones múltiples

In [ ]:
df.group_by('target').agg([
    pl.col('sepal length (cm)').mean().alias('mean_sepal_len'),
    pl.col('sepal length (cm)').min().alias('min_sepal_len'),
    pl.col('sepal length (cm)').max().alias('max_sepal_len'),
    pl.col('petal length (cm)').std().alias('std_petal_len')
])

### Explicación
Agrupamos por `target` y aplicamos múltiples agregaciones usando expresiones: cada `pl.col(...).func().alias()` define una métrica clara y eficiente (lazy-friendly).


## 4. Visualización por grupos (listas nativas)


In [ ]:
for group in df.partition_by('target', as_dict=True).values():
    label = group['target'][0]
    plt.figure()
    plt.hist(group['petal length (cm)'].to_list())
    plt.title(f'Histograma - Petal Length (target={label})')


### Explicación
`partition_by` separa grupos en DataFrames Polars y `to_list()` entrega únicamente datos nativos a Matplotlib, sin conversión intermedia a pandas.


## 5. Columnas derivadas con expresiones Polars

In [ ]:
df = df.with_columns([
    (pl.col('sepal length (cm)') / pl.col('sepal width (cm)')).alias('sepal_ratio')
])
df.head()

### Explicación
`with_columns` crea columna derivada `sepal_ratio` usando expresión vectorizada Polars (sin bucles) combinando largo/ancho del sépalo.


## 6. Join con dataset artificial

In [ ]:
df_extra = pl.DataFrame({
    'target':[0,1,2],
    'species':['setosa','versicolor','virginica']
})

df_merged = df.join(df_extra, on='target')
df_merged.head()

### Explicación
`join` interno por `target` añade nombre de especie desde tabla auxiliar. Polars mantiene tipos y evita duplicados si claves son únicas.


## 7. Resampling temporal (simulado)

In [ ]:
df.height


In [ ]:
from datetime import date, timedelta

# Las fechas se generan con la biblioteca estándar y permanecen en Polars.
rng = [date(2024, 1, 1) + timedelta(days=i) for i in range(df.height)]
df_time = df.with_columns(pl.Series('fecha', rng))
df_time.group_by_dynamic('fecha', every='1w', period='1w', closed='left').agg(
    pl.col('petal length (cm)').mean().alias('mean_petal_length')
).head()


### Explicación
El muestreo semanal se realiza nativamente con `group_by_dynamic`; no hace falta convertir a pandas. La conversión solo sería necesaria para interoperar con una API externa que la exigiera.


## 8. Rolling window nativa de Polars


In [ ]:
df_time = df_time.with_columns(
    pl.col('petal length (cm)').rolling_mean(window_size=7).alias('rolling_petal')
)
df_time.select(['petal length (cm)', 'rolling_petal']).head(15)


### Explicación
`rolling_mean` calcula la media móvil directamente con expresiones Polars, sin depender de pandas.


### Gráfico rolling

In [ ]:
plt.plot(df_time['petal length (cm)'].to_list())
plt.plot(df_time['rolling_petal'].to_list())
plt.title('Petal Length vs Rolling Mean 7d')
plt.legend(['Original','Rolling'])


### Explicación
Gráfico comparando serie original y su media móvil suavizada para observar tendencia y reducir ruido en la variable de pétalo.


## 9. Intervalos e interval joins (Polars estilo)

In [ ]:
df_vals = pl.DataFrame({'valor': [round((i * 1.37) % 8, 2) for i in range(10)]})


df_int = pl.DataFrame({
    'left':[0,2,4,6],
    'right':[2,4,6,8],
    'categoria':['baja','media','alta','max']
})

result = (
    df_vals.lazy()
    .join(
        df_int.lazy(),
        how="cross"
    )
    .filter(
        (pl.col("valor") >= pl.col("left")) &
        (pl.col("valor") <  pl.col("right"))
    )
    .collect()
)

result

### Explicación
Simulación de interval join: se genera tabla de intervalos (`left`, `right`) y se hace un `join` condicional (`predicate='<=')`. Polars permite condiciones en joins para emparejamientos más flexibles.


## 10. Optimización de tipos en Polars

In [ ]:
df_opt = df.with_columns([
    pl.col('target').cast(pl.String).cast(pl.Categorical)
])
df_opt.dtypes

### Explicación
Conversión de `target` a categoría (`pl.Categorical`) optimiza memoria y puede acelerar agrupaciones y comparaciones en Polars.
